# Hybrid RAG with Re-Ranking — Self-Contained Demo

Everything needed to run the demo lives in **this notebook**.
No dependency on the `src/` folder — just the on-disk data and indexes.

### What you need locally
```
data/corpus/chunks.jsonl
indexes/dense/{embeddings.npy, faiss.index, dense_meta.json}
indexes/bm25/{bm25_model.pkl, tokenized_docs.pkl, bm25_meta.json}
```

### Python packages
```
pip install numpy faiss-cpu rank_bm25 sentence-transformers transformers torch
```

### Structure of this notebook
1. Imports & data classes
2. Load chunks, dense index, BM25 index
3. Dense retriever (inlined)
4. Sparse BM25 retriever (inlined)
5. Reciprocal Rank Fusion (inlined)
6. Cross-encoder re-ranker (inlined)
7. LLM generator + prompt builder (inlined)
8. End-to-end HybridRAG pipeline (inlined)
9. Demo — with/without reranker
10. Stage-by-stage inspection
11. Free-form query cell

## 1. Imports and data classes

In [1]:
import json, os, pickle, time
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np


@dataclass(frozen=True)
class Chunk:
    chunk_id: str
    url: str
    title: str
    chunk_index: int
    text: str


@dataclass(frozen=True)
class RetrievedChunk:
    chunk: Chunk
    dense_score: float = 0.0
    bm25_score: float = 0.0
    rrf_score: float = 0.0
    dense_rank: Optional[int] = None
    bm25_rank: Optional[int] = None
    rerank_score: Optional[float] = None
    rerank_rank: Optional[int] = None


@dataclass(frozen=True)
class RAGAnswer:
    query: str
    answer: str
    context_chunks: List[RetrievedChunk]
    latency_ms: Dict[str, float]
    debug: Dict[str, Any] = field(default_factory=dict)

print('Data classes ready')

Data classes ready


## 2. Load chunks and set paths

In [2]:
PROJECT_ROOT = Path.cwd()
# Walk up until we find a project marker (requirements.txt)
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'requirements.txt').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
CHUNKS_PATH  = PROJECT_ROOT / 'data'    / 'corpus' / 'chunks.jsonl'
DENSE_DIR    = PROJECT_ROOT / 'indexes' / 'dense'
BM25_DIR     = PROJECT_ROOT / 'indexes' / 'bm25'

def read_jsonl(path: Path) -> List[dict]:
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

chunks: List[Chunk] = [Chunk(**r) for r in read_jsonl(CHUNKS_PATH)]
print(f'Loaded {len(chunks):,} chunks')
print(f'Example title: {chunks[0].title!r}')

Loaded 9,083 chunks


Example title: 'Paris'


## 3. Dense retriever

Encode the query with **sentence-transformers/all-mpnet-base-v2** and match it against a FAISS cosine-similarity index over the pre-computed chunk embeddings.

In [3]:
from sentence_transformers import SentenceTransformer
import faiss

def _normalize(matrix: np.ndarray) -> np.ndarray:
    return matrix / (np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-12)


class DenseIndex:
    def __init__(self, model_name: str = 'all-mpnet-base-v2'):
        self.model_name = model_name
        self.encoder: Optional[SentenceTransformer] = None
        self.embeddings: Optional[np.ndarray] = None
        self.faiss_index = None

    @classmethod
    def load(cls, input_dir: Path) -> 'DenseIndex':
        with open(input_dir / 'dense_meta.json', 'r', encoding='utf-8') as f:
            meta = json.load(f)
        inst = cls(model_name=meta['model_name'])
        inst.encoder = SentenceTransformer(inst.model_name)
        inst.embeddings = np.load(input_dir / 'embeddings.npy')
        faiss_path = input_dir / 'faiss.index'
        if faiss_path.exists():
            inst.faiss_index = faiss.read_index(str(faiss_path))
        return inst

    def search(self, query: str, top_k: int = 50) -> Tuple[np.ndarray, np.ndarray]:
        q = self.encoder.encode([query])
        q = _normalize(np.asarray(q, dtype=np.float32))
        if self.faiss_index is not None:
            scores, idx = self.faiss_index.search(q, top_k)
            return idx[0], scores[0]
        scores = np.dot(self.embeddings, q[0])
        ranked = np.argsort(-scores)[:top_k]
        return ranked, scores[ranked]


dense_index = DenseIndex.load(DENSE_DIR)
print('Dense index loaded -', dense_index.embeddings.shape)

Dense index loaded - (9083, 768)


## 4. Sparse retriever — BM25

Classic term-frequency / IDF ranking via `rank_bm25`. The pickled model in `indexes/bm25/` was trained on tokenized chunks.

In [4]:
from rank_bm25 import BM25Okapi

def simple_tokenize(text: str) -> List[str]:
    return [t for t in text.lower().split() if t]


class BM25Index:
    def __init__(self):
        self.model: Optional[BM25Okapi] = None
        self.tokenized_documents: List[List[str]] = []

    @classmethod
    def load(cls, input_dir: Path) -> 'BM25Index':
        inst = cls()
        with open(input_dir / 'bm25_model.pkl', 'rb') as f:
            inst.model = pickle.load(f)
        with open(input_dir / 'tokenized_docs.pkl', 'rb') as f:
            inst.tokenized_documents = pickle.load(f)
        return inst

    def search(self, query: str, top_k: int = 50) -> Tuple[np.ndarray, np.ndarray]:
        tokens = simple_tokenize(query)
        scores = np.asarray(self.model.get_scores(tokens), dtype=np.float32)
        ranked = np.argsort(-scores)[:top_k]
        return ranked, scores[ranked]


bm25_index = BM25Index.load(BM25_DIR)
print('BM25 index loaded -', len(bm25_index.tokenized_documents), 'documents')

BM25 index loaded - 9083 documents


## 5. Reciprocal Rank Fusion

Combines the dense and sparse ranking lists. No score calibration required — it only uses the **rank position**.

$$ \text{RRF}(d) = \sum_i \frac{1}{k + \text{rank}_i(d)}, \quad k = 60 $$

In [5]:
def reciprocal_rank_fusion(dense_idx: np.ndarray,
                           sparse_idx: np.ndarray,
                           fusion_constant: int = 60,
                           limit: Optional[int] = None) -> Dict[int, float]:
    if limit is not None:
        dense_idx  = dense_idx[:limit]
        sparse_idx = sparse_idx[:limit]
    fused: Dict[int, float] = {}
    for rank, doc in enumerate(dense_idx, start=1):
        fused[int(doc)] = fused.get(int(doc), 0.0) + 1.0 / (fusion_constant + rank)
    for rank, doc in enumerate(sparse_idx, start=1):
        fused[int(doc)] = fused.get(int(doc), 0.0) + 1.0 / (fusion_constant + rank)
    return fused


def top_n_by_score(score_map: Dict[int, float], top_n: int) -> List[Tuple[int, float]]:
    return sorted(score_map.items(), key=lambda kv: kv[1], reverse=True)[:top_n]

## 6. Cross-encoder re-ranker

The reranker scores each `(query, chunk)` pair **jointly** instead of separately — slower per-pair but much more accurate. We run it only on the top 25 fused candidates.

**First call downloads ~80 MB.**

In [6]:
from sentence_transformers import CrossEncoder

class CrossEncoderReranker:
    def __init__(self, model_name: str = 'cross-encoder/ms-marco-MiniLM-L-6-v2',
                 device: str = 'cpu'):
        self.model_name = model_name
        self.device = device
        self._model: Optional[CrossEncoder] = None

    def load(self) -> None:
        if self._model is None:
            self._model = CrossEncoder(self.model_name, device=self.device)

    def rerank(self, query: str, candidates: List[RetrievedChunk],
               top_n: int) -> List[RetrievedChunk]:
        if not candidates:
            return []
        self.load()
        pairs  = [[query, c.chunk.text] for c in candidates]
        scores = self._model.predict(pairs)
        ranked = sorted(zip(candidates, [float(s) for s in scores]),
                        key=lambda x: x[1], reverse=True)
        return [replace(c, rerank_score=s, rerank_rank=i + 1)
                for i, (c, s) in enumerate(ranked[:top_n])]


reranker = CrossEncoderReranker()
reranker.load()
print('Reranker loaded')

Reranker loaded


## 7. LLM generator + prompt builder

`google/flan-t5-base` as the generator — small, CPU-friendly, instruction-tuned.

In [7]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

class LLMGenerator:
    def __init__(self, model_name: str = 'google/flan-t5-base', device: str = 'cpu'):
        self.model_name = model_name
        self.device = device
        self.model = None
        self.tokenizer = None

    def load(self) -> None:
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
        self.model.to(self.device)

    def generate(self, prompt: str, max_new_tokens: int = 128) -> str:
        inputs = self.tokenizer(prompt, return_tensors='pt', truncation=True)
        inputs = {k: v.to(self.device) for k, v in inputs.items()}
        with torch.no_grad():
            out = self.model.generate(**inputs,
                                      max_new_tokens=max_new_tokens,
                                      do_sample=False)
        return self.tokenizer.decode(out[0], skip_special_tokens=True)


def build_prompt(query: str, context_chunks: List[RetrievedChunk]) -> str:
    blocks = []
    for i, rc in enumerate(context_chunks, start=1):
        blocks.append(f'Source {i}: {rc.chunk.title} ({rc.chunk.url})\n{rc.chunk.text}')
    return (
        'You are a helpful assistant. Answer the question using only the provided context. '
        'If the question has multiple parts, address every part in one complete sentence. '
        'Include all relevant facts such as locations, dates, years, and numbers. '
        'If the answer is not present in the context, say you do not know.\n\n'
        f'Question: {query}\n\n'
        f'Context:\n' + '\n\n'.join(blocks) + '\n\n'
        'Complete answer:'
    )


generator = LLMGenerator()
generator.load()
print('Flan-T5-base loaded')

Flan-T5-base loaded


## 8. End-to-end HybridRAG pipeline

Dense + BM25 → RRF → (optional cross-encoder rerank) → Flan-T5.

In [8]:
class HybridRAG:
    def __init__(self, chunks, dense_index, bm25_index, generator,
                 rrf_constant: int = 60,
                 reranker: Optional[CrossEncoderReranker] = None):
        self.chunks = chunks
        self.dense_index = dense_index
        self.bm25_index = bm25_index
        self.generator = generator
        self.rrf_constant = rrf_constant
        self.reranker = reranker

    def retrieve(self, query: str, top_k: int = 50, context_size: int = 6,
                 use_reranker: bool = False,
                 rerank_pool: int = 25) -> List[RetrievedChunk]:
        dense_idx, dense_sc = self.dense_index.search(query, top_k)
        sparse_idx, sparse_sc = self.bm25_index.search(query, top_k)

        fused = reciprocal_rank_fusion(dense_idx, sparse_idx,
                                       fusion_constant=self.rrf_constant,
                                       limit=top_k)

        rerank_active = use_reranker and self.reranker is not None
        first_n = max(rerank_pool, context_size) if rerank_active else context_size
        fused_top = top_n_by_score(fused, first_n)

        d_rank = {int(d): i + 1 for i, d in enumerate(dense_idx)}
        s_rank = {int(d): i + 1 for i, d in enumerate(sparse_idx)}
        d_sc   = {int(d): float(s) for d, s in zip(dense_idx, dense_sc)}
        s_sc   = {int(d): float(s) for d, s in zip(sparse_idx, sparse_sc)}

        retrieved: List[RetrievedChunk] = []
        for doc_idx, rrf_s in fused_top:
            retrieved.append(RetrievedChunk(
                chunk=self.chunks[int(doc_idx)],
                dense_score=d_sc.get(int(doc_idx), 0.0),
                bm25_score=s_sc.get(int(doc_idx), 0.0),
                rrf_score=float(rrf_s),
                dense_rank=d_rank.get(int(doc_idx)),
                bm25_rank=s_rank.get(int(doc_idx)),
            ))

        if rerank_active:
            retrieved = self.reranker.rerank(query, retrieved, top_n=context_size)
        return retrieved

    def answer(self, query: str, top_k: int = 50, context_size: int = 6,
               max_new_tokens: int = 160,
               use_reranker: bool = False,
               rerank_pool: int = 25) -> RAGAnswer:
        t0 = time.perf_counter()
        tr0 = time.perf_counter()
        ctx = self.retrieve(query, top_k, context_size,
                            use_reranker=use_reranker, rerank_pool=rerank_pool)
        tr1 = time.perf_counter()
        prompt = build_prompt(query, ctx)
        tg0 = time.perf_counter()
        out  = self.generator.generate(prompt, max_new_tokens=max_new_tokens)
        tg1 = time.perf_counter()
        t1   = time.perf_counter()
        return RAGAnswer(
            query=query, answer=out, context_chunks=ctx,
            latency_ms={
                'retrieve_total': (tr1 - tr0) * 1000.0,
                'generate':       (tg1 - tg0) * 1000.0,
                'total':          (t1 - t0) * 1000.0,
            },
        )


rag = HybridRAG(chunks=chunks,
                dense_index=dense_index,
                bm25_index=bm25_index,
                generator=generator,
                reranker=reranker)
print('HybridRAG ready')

HybridRAG ready


## 9. Pretty-printer

In [9]:
def show(result: RAGAnswer, max_sources: int = 5, snippet_chars: int = 220):
    print('=' * 80)
    print(f'Q: {result.query}')
    print('-' * 80)
    print(f'A: {result.answer}')
    print('-' * 80)
    lat = result.latency_ms
    print(f"Latency  retrieve={lat['retrieve_total']:.0f}ms  "
          f"generate={lat['generate']:.0f}ms  total={lat['total']:.0f}ms")
    print('-' * 80)
    print('Top sources:')
    for i, c in enumerate(result.context_chunks[:max_sources], start=1):
        rr = f"  rerank={c.rerank_score:.3f}" if c.rerank_score is not None else ''
        print(f'  {i}. {c.chunk.title}')
        print(f'     dense_rank={c.dense_rank}  bm25_rank={c.bm25_rank}  '
              f'rrf={c.rrf_score:.4f}{rr}')
        snippet = c.chunk.text[:snippet_chars].replace('\n', ' ')
        print(f'     {snippet}...')
    print('=' * 80)
    print()

## 10. Demo — without and with reranker

Same query, same context size, only the reranker toggled.

In [10]:
query = 'When was the Eiffel Tower built?'

print('>>> WITHOUT reranker')
show(rag.answer(query, use_reranker=False))

print('>>> WITH reranker')
show(rag.answer(query, use_reranker=True, rerank_pool=25))

>>> WITHOUT reranker


Q: When was the Eiffel Tower built?
--------------------------------------------------------------------------------
A: 1887 to 1889.
--------------------------------------------------------------------------------
Latency  retrieve=320ms  generate=2076ms  total=2397ms
--------------------------------------------------------------------------------
Top sources:
  1. Eiffel Tower
     dense_rank=1  bm25_rank=4  rrf=0.0320
     The Eiffel Tower ( / ˈ aɪ f əl / ⓘ EYE -fəl ; French : Tour Eiffel [tuʁ ɛfɛl] ⓘ ) is a lattice tower on the Champ de Mars in Paris , France. It is named after the engineer Gustave Eiffel , whose company designed and buil...
  2. Eiffel Tower
     dense_rank=7  bm25_rank=1  rrf=0.0313
     an aerial, exchanged wireless signals with the United States Naval Observatory , which used an aerial in Arlington County, Virginia . The object of the transmissions was to measure the difference in longitude between Par...
  3. Eiffel Tower
     dense_rank=8  bm25_rank=2  rrf=0.

Q: When was the Eiffel Tower built?
--------------------------------------------------------------------------------
A: 1887 to 1889.
--------------------------------------------------------------------------------
Latency  retrieve=1400ms  generate=759ms  total=2160ms
--------------------------------------------------------------------------------
Top sources:
  1. Eiffel Tower
     dense_rank=1  bm25_rank=4  rrf=0.0320  rerank=7.429
     The Eiffel Tower ( / ˈ aɪ f əl / ⓘ EYE -fəl ; French : Tour Eiffel [tuʁ ɛfɛl] ⓘ ) is a lattice tower on the Champ de Mars in Paris , France. It is named after the engineer Gustave Eiffel , whose company designed and buil...
  2. Eiffel Tower
     dense_rank=6  bm25_rank=8  rrf=0.0299  rerank=5.176
     tower, he finished his talk by saying the tower would symbolise [n]ot only the art of the modern engineer, but also the century of Industry and Science in which we are living, and for which the way was prepared by the gr...
  3. Eiffel Tower
     dense

## 11. Five demo queries, side by side

In [11]:
demo_queries = [
    'When was the Eiffel Tower built?',
    'What is the height of Burj Khalifa?',
    'Which country is Machu Picchu located in?',
    'What is the Great Barrier Reef made of?',
    'Who designed the Sagrada Familia?',
]

for q in demo_queries:
    print('\n>>> WITHOUT reranker')
    show(rag.answer(q, use_reranker=False),
         max_sources=3, snippet_chars=120)
    print('>>> WITH reranker')
    show(rag.answer(q, use_reranker=True, rerank_pool=25),
         max_sources=3, snippet_chars=120)


>>> WITHOUT reranker


Q: When was the Eiffel Tower built?
--------------------------------------------------------------------------------
A: 1887 to 1889.
--------------------------------------------------------------------------------
Latency  retrieve=65ms  generate=657ms  total=722ms
--------------------------------------------------------------------------------
Top sources:
  1. Eiffel Tower
     dense_rank=1  bm25_rank=4  rrf=0.0320
     The Eiffel Tower ( / ˈ aɪ f əl / ⓘ EYE -fəl ; French : Tour Eiffel [tuʁ ɛfɛl] ⓘ ) is a lattice tower on the Champ de Mar...
  2. Eiffel Tower
     dense_rank=7  bm25_rank=1  rrf=0.0313
     an aerial, exchanged wireless signals with the United States Naval Observatory , which used an aerial in Arlington Count...
  3. Eiffel Tower
     dense_rank=8  bm25_rank=2  rrf=0.0308
     the tower made it a beacon in Paris's night sky, and 20,000 flashing bulbs gave the tower a sparkly appearance for five ...

>>> WITH reranker


Q: When was the Eiffel Tower built?
--------------------------------------------------------------------------------
A: 1887 to 1889.
--------------------------------------------------------------------------------
Latency  retrieve=1129ms  generate=819ms  total=1948ms
--------------------------------------------------------------------------------
Top sources:
  1. Eiffel Tower
     dense_rank=1  bm25_rank=4  rrf=0.0320  rerank=7.429
     The Eiffel Tower ( / ˈ aɪ f əl / ⓘ EYE -fəl ; French : Tour Eiffel [tuʁ ɛfɛl] ⓘ ) is a lattice tower on the Champ de Mar...
  2. Eiffel Tower
     dense_rank=6  bm25_rank=8  rrf=0.0299  rerank=5.176
     tower, he finished his talk by saying the tower would symbolise [n]ot only the art of the modern engineer, but also the ...
  3. Eiffel Tower
     dense_rank=12  bm25_rank=6  rrf=0.0290  rerank=4.973
     of Paris which happen to include the lit tower. The author of the 1989 lighting display, lighting director Pierre Bideau...


>>> WITHOUT reranker


Q: What is the height of Burj Khalifa?
--------------------------------------------------------------------------------
A: 829.8 m (2,722 ft, or just over half a mile) and a roof height (excluding the antenna, but including a 242.6 m spire) [ 2 ] of 828 m (2,717 ft).
--------------------------------------------------------------------------------
Latency  retrieve=86ms  generate=2573ms  total=2660ms
--------------------------------------------------------------------------------
Top sources:
  1. Burj Khalifa
     dense_rank=1  bm25_rank=2  rrf=0.0325
     The Burj Khalifa [ a ] (known as the Burj Dubai before its inauguration) is a megatall skyscraper in Dubai , United Arab...
  2. Burj Khalifa
     dense_rank=4  bm25_rank=3  rrf=0.0315
     selected for their expertise in structural and MEP (mechanical, electrical and plumbing) engineering. [ 39 ] Hyder Consu...
  3. Burj Khalifa
     dense_rank=2  bm25_rank=6  rrf=0.0313
     attached to the 160th floor at 672 m (2,205 ft). The two 

Q: What is the height of Burj Khalifa?
--------------------------------------------------------------------------------
A: 829.8 m (2,722 ft, or just over half a mile) and a roof height (excluding the antenna, but including a 242.6 m spire) [ 2 ] of 828 m (2,717 ft).
--------------------------------------------------------------------------------
Latency  retrieve=1279ms  generate=3874ms  total=5153ms
--------------------------------------------------------------------------------
Top sources:
  1. Burj Khalifa
     dense_rank=1  bm25_rank=2  rrf=0.0325  rerank=7.578
     The Burj Khalifa [ a ] (known as the Burj Dubai before its inauguration) is a megatall skyscraper in Dubai , United Arab...
  2. CN Tower
     dense_rank=5  bm25_rank=13  rrf=0.0291  rerank=6.291
     are numerous radio masts and towers , which are held in place by guy-wires , the tallest being the KVLY-TV mast in Blanc...
  3. Burj Al Arab
     dense_rank=14  bm25_rank=4  rrf=0.0291  rerank=5.122
     contains over 7

...

Q: Which country is Machu Picchu located in?
--------------------------------------------------------------------------------
A: Peru.
--------------------------------------------------------------------------------
Latency  retrieve=3608ms  generate=1767ms  total=5374ms
--------------------------------------------------------------------------------
Top sources:
  1. Machu Picchu
     dense_rank=2  bm25_rank=1  rrf=0.0325  rerank=7.724
     Machu Picchu [ b ] is a 15th-century Inca citadel located in the Eastern Cordillera of southern Peru on a mountain ridge...
  2. Machu Picchu
     dense_rank=1  bm25_rank=3  rrf=0.0323  rerank=5.272
     [ 130 ] [ 131 ] Due to these disruptions, the Ministry of Culture closed the site indefinitely on January 22, 2023, and ...
  3. Machu Picchu
     dense_rank=5  bm25_rank=4  rrf=0.0310  rerank=4.224
     which arrives every morning from Cusco and returns every afternoon. There is also a luxury hotel on the mountain, near t...


>>> WITHOUT reranker

Q: What is the Great Barrier Reef made of?
--------------------------------------------------------------------------------
A: coral.
--------------------------------------------------------------------------------
Latency  retrieve=223ms  generate=1719ms  total=1942ms
--------------------------------------------------------------------------------
Top sources:
  1. Great Barrier Reef
     dense_rank=3  bm25_rank=2  rrf=0.0320
     The Australian Institute of Marine Science conducts annual surveys of the Great Barrier Reef's status, and the 2022 repo...
  2. Coral reef
     dense_rank=5  bm25_rank=1  rrf=0.0318
     reef and the land. A barrier reef can encircle an island, and once the island sinks below sea level, a roughly circular ...
  3. Great Barrier Reef
     dense_rank=1  bm25_rank=7  rrf=0.0313
     The Great Barrier Reef is the world's largest coral reef system, [ 1 ] [ 2 ] composed of over 2,900 individual reefs [ 3...

>>> WITH reranker


Q: What is the Great Barrier Reef made of?
--------------------------------------------------------------------------------
A: coral polyps.
--------------------------------------------------------------------------------
Latency  retrieve=3596ms  generate=2213ms  total=5809ms
--------------------------------------------------------------------------------
Top sources:
  1. Great Barrier Reef
     dense_rank=1  bm25_rank=7  rrf=0.0313  rerank=6.595
     The Great Barrier Reef is the world's largest coral reef system, [ 1 ] [ 2 ] composed of over 2,900 individual reefs [ 3...
  2. Great Barrier Reef
     dense_rank=4  bm25_rank=12  rrf=0.0295  rerank=1.215
     The Great Barrier Reef has long been known to and used by the Aboriginal Australian and Torres Strait Islander peoples, ...
  3. Great Barrier Reef
     dense_rank=3  bm25_rank=2  rrf=0.0320  rerank=0.992
     The Australian Institute of Marine Science conducts annual surveys of the Great Barrier Reef's status, and the 2022 repo.

Q: Who designed the Sagrada Familia?
--------------------------------------------------------------------------------
A: Gaud.
--------------------------------------------------------------------------------
Latency  retrieve=172ms  generate=2071ms  total=2243ms
--------------------------------------------------------------------------------
Top sources:
  1. Sagrada Família
     dense_rank=3  bm25_rank=1  rrf=0.0323
     15 and 25 percent complete. [ 10 ] [ 23 ] After Gaudí's death, work continued under the direction of his main disciple D...
  2. Sagrada Família
     dense_rank=1  bm25_rank=6  rrf=0.0315
     stopped and the basilica was closed. [ 47 ] This was the first time the construction had been halted since the Spanish C...
  3. Sagrada Família
     dense_rank=5  bm25_rank=2  rrf=0.0315
     Basílica i Temple Expiatori de la Sagrada Família , [ a ] or simply Sagrada Família , is a church under construction in ...

>>> WITH reranker


Q: Who designed the Sagrada Familia?
--------------------------------------------------------------------------------
A: Antoni Gaud.
--------------------------------------------------------------------------------
Latency  retrieve=3460ms  generate=2109ms  total=5569ms
--------------------------------------------------------------------------------
Top sources:
  1. Sagrada Família
     dense_rank=5  bm25_rank=2  rrf=0.0315  rerank=8.271
     Basílica i Temple Expiatori de la Sagrada Família , [ a ] or simply Sagrada Família , is a church under construction in ...
  2. Sagrada Família
     dense_rank=2  bm25_rank=8  rrf=0.0308  rerank=5.819
     the building in 2026, though the announcement stated that work on sculptures, decorative details and a controversial sta...
  3. Sagrada Família
     dense_rank=3  bm25_rank=1  rrf=0.0323  rerank=5.280
     15 and 25 percent complete. [ 10 ] [ 23 ] After Gaudí's death, work continued under the direction of his main disciple D...



## 12. Compound query — the reranker's moment

In [12]:
compound = 'Where is the Burj Khalifa and when was it built?'

print('>>> WITHOUT reranker')
show(rag.answer(compound, use_reranker=False))

print('>>> WITH reranker')
show(rag.answer(compound, use_reranker=True, rerank_pool=25))

>>> WITHOUT reranker


Q: Where is the Burj Khalifa and when was it built?
--------------------------------------------------------------------------------
A: Construction of the Burj Khalifa began in 2004; the exterior was completed five years later.
--------------------------------------------------------------------------------
Latency  retrieve=221ms  generate=2027ms  total=2248ms
--------------------------------------------------------------------------------
Top sources:
  1. Burj Khalifa
     dense_rank=1  bm25_rank=2  rrf=0.0325
     The Burj Khalifa [ a ] (known as the Burj Dubai before its inauguration) is a megatall skyscraper in Dubai , United Arab Emirates. Designed by Skidmore, Owings & Merrill , it is the world's tallest structure , with a tot...
  2. Burj Khalifa
     dense_rank=2  bm25_rank=5  rrf=0.0315
     spot. [ 71 ] On 8 February 2010, the observation deck was closed to the public for two months after power-supply problems caused an elevator to become stuck between floors, trapping a g

Q: Where is the Burj Khalifa and when was it built?
--------------------------------------------------------------------------------
A: Construction of the Burj Khalifa began in 2004; the exterior was completed five years later.
--------------------------------------------------------------------------------
Latency  retrieve=1484ms  generate=1129ms  total=2613ms
--------------------------------------------------------------------------------
Top sources:
  1. Burj Khalifa
     dense_rank=1  bm25_rank=2  rrf=0.0325  rerank=6.944
     The Burj Khalifa [ a ] (known as the Burj Dubai before its inauguration) is a megatall skyscraper in Dubai , United Arab Emirates. Designed by Skidmore, Owings & Merrill , it is the world's tallest structure , with a tot...
  2. Dubai
     dense_rank=3  bm25_rank=10  rrf=0.0302  rerank=5.139
     ] The completion of the Burj Khalifa, following the construction boom that began in the 1980s, accelerated in the 1990s and reached a rapid pace during the 2000s,

## 13. Stage-by-stage inspection

Raw dense top-10, raw BM25 top-10, and the fused RRF top-10.

In [13]:
q = 'What is the height of Burj Khalifa?'

dense_idx, dense_sc = dense_index.search(q, top_k=10)
bm25_idx,  bm25_sc  = bm25_index.search(q,  top_k=10)

print('Top 10 - Dense')
for i, (d, s) in enumerate(zip(dense_idx, dense_sc), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  score={s:.3f}')

print('\nTop 10 - BM25')
for i, (d, s) in enumerate(zip(bm25_idx, bm25_sc), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  score={s:.3f}')

fused = reciprocal_rank_fusion(dense_idx, bm25_idx, fusion_constant=60)
print('\nTop 10 - Fused (RRF)')
for i, (d, s) in enumerate(top_n_by_score(fused, 10), 1):
    print(f'  {i:2d}. {chunks[int(d)].title:<40}  rrf={s:.4f}')

Top 10 - Dense
   1. Burj Khalifa                              score=0.736
   2. Burj Khalifa                              score=0.707
   3. Burj Khalifa                              score=0.687
   4. Burj Khalifa                              score=0.686
   5. CN Tower                                  score=0.682
   6. Dubai                                     score=0.680
   7. Burj Khalifa                              score=0.661
   8. Burj Khalifa                              score=0.658
   9. Burj Khalifa                              score=0.647
  10. Burj Khalifa                              score=0.639

Top 10 - BM25
   1. Burj Al Arab                              score=29.940
   2. Burj Khalifa                              score=29.460
   3. Burj Khalifa                              score=28.153
   4. Burj Al Arab                              score=27.715
   5. Burj Khalifa                              score=25.122
   6. Burj Khalifa                              score=24.711
   7

## 14. Tune rerank pool on the fly

In [14]:
q = 'Who designed the Sagrada Familia?'

for pool in [10, 25, 40]:
    print(f'\n--- rerank_pool = {pool} ---')
    show(rag.answer(q, context_size=4, use_reranker=True, rerank_pool=pool),
         max_sources=4, snippet_chars=100)


--- rerank_pool = 10 ---


Q: Who designed the Sagrada Familia?
--------------------------------------------------------------------------------
A: Antoni Gaud.
--------------------------------------------------------------------------------
Latency  retrieve=563ms  generate=636ms  total=1199ms
--------------------------------------------------------------------------------
Top sources:
  1. Sagrada Família
     dense_rank=5  bm25_rank=2  rrf=0.0315  rerank=8.271
     Basílica i Temple Expiatori de la Sagrada Família , [ a ] or simply Sagrada Família , is a church un...
  2. Sagrada Família
     dense_rank=2  bm25_rank=8  rrf=0.0308  rerank=5.819
     the building in 2026, though the announcement stated that work on sculptures, decorative details and...
  3. Sagrada Família
     dense_rank=3  bm25_rank=1  rrf=0.0323  rerank=5.280
     15 and 25 percent complete. [ 10 ] [ 23 ] After Gaudí's death, work continued under the direction of...
  4. Sagrada Família
     dense_rank=7  bm25_rank=7  rrf=0.0299  rerank=4.07

Q: Who designed the Sagrada Familia?
--------------------------------------------------------------------------------
A: Antoni Gaud.
--------------------------------------------------------------------------------
Latency  retrieve=1088ms  generate=580ms  total=1668ms
--------------------------------------------------------------------------------
Top sources:
  1. Sagrada Família
     dense_rank=5  bm25_rank=2  rrf=0.0315  rerank=8.271
     Basílica i Temple Expiatori de la Sagrada Família , [ a ] or simply Sagrada Família , is a church un...
  2. Sagrada Família
     dense_rank=2  bm25_rank=8  rrf=0.0308  rerank=5.819
     the building in 2026, though the announcement stated that work on sculptures, decorative details and...
  3. Sagrada Família
     dense_rank=3  bm25_rank=1  rrf=0.0323  rerank=5.280
     15 and 25 percent complete. [ 10 ] [ 23 ] After Gaudí's death, work continued under the direction of...
  4. Sagrada Família
     dense_rank=7  bm25_rank=7  rrf=0.0299  rerank=4.0

Q: Who designed the Sagrada Familia?
--------------------------------------------------------------------------------
A: Antoni Gaud.
--------------------------------------------------------------------------------
Latency  retrieve=1598ms  generate=603ms  total=2200ms
--------------------------------------------------------------------------------
Top sources:
  1. Sagrada Família
     dense_rank=5  bm25_rank=2  rrf=0.0315  rerank=8.271
     Basílica i Temple Expiatori de la Sagrada Família , [ a ] or simply Sagrada Família , is a church un...
  2. Sagrada Família
     dense_rank=2  bm25_rank=8  rrf=0.0308  rerank=5.819
     the building in 2026, though the announcement stated that work on sculptures, decorative details and...
  3. Sagrada Família
     dense_rank=3  bm25_rank=1  rrf=0.0323  rerank=5.280
     15 and 25 percent complete. [ 10 ] [ 23 ] After Gaudí's death, work continued under the direction of...
  4. Sagrada Família
     dense_rank=7  bm25_rank=7  rrf=0.0299  rerank=4.0

## 15. Free-form query — audience suggestions

In [15]:
my_query = 'What is the Colosseum in Rome known for?'
show(rag.answer(my_query, use_reranker=True, rerank_pool=25))

Q: What is the Colosseum in Rome known for?
--------------------------------------------------------------------------------
A: The largest ancient amphitheatre ever built, and is the largest standing amphitheatre in the world.
--------------------------------------------------------------------------------
Latency  retrieve=1134ms  generate=1476ms  total=2610ms
--------------------------------------------------------------------------------
Top sources:
  1. Colosseum
     dense_rank=1  bm25_rank=3  rrf=0.0323  rerank=7.375
     The Colosseum ( / ˌ k ɒ . l ə . ˈ s iː . ə m / KOL -ə- SEE -əm ; Italian : Colosseo [kolosˈsɛːo] , ultimately from Ancient Greek word "kolossos" meaning a large statue or giant) is an elliptical amphitheatre in the centr...
  2. Colosseum
     dense_rank=2  bm25_rank=8  rrf=0.0308  rerank=4.615
     the stonework together were pried or hacked out of the walls, leaving numerous pockmarks which still scar the building today. During the 16th and 17th century, Chu

---
**Done.** All logic is inlined in this notebook — nothing is imported from `src/`.